In [1]:
import sys, os
# Erzwingt den Hauptpfad
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.db_connect import load_sql, get_engine

In [2]:
import pandas as pd
from src.db_connect import get_engine, text

engine = get_engine()
# Diese Abfrage liefert die menschenlesbare Größe der gesamten Datenbank
query = "SELECT pg_size_pretty(pg_database_size('mimic')) AS groesse_auf_disk;"

with engine.connect() as conn:
    res = pd.read_sql(text(query), conn)
    
print(f"Peer muss mindestens so viel Platz freimachen: {res['groesse_auf_disk'][0]}")

Peer muss mindestens so viel Platz freimachen: 45 GB


In [7]:
from sqlalchemy import text
import pandas as pd

# Abfrage: Wer erhielt Noradrenalin (Norepinephrine)?
query_vaso_check = text("""
SELECT 
    c.gender,
    COUNT(DISTINCT c.hadm_id) as gesamt_patienten,
    COUNT(DISTINCT v.hadm_id) as patienten_mit_vaso,
    ROUND(100.0 * COUNT(DISTINCT v.hadm_id) / COUNT(DISTINCT c.hadm_id), 2) as vaso_rate_prozent
FROM cohort_respiratory c
LEFT JOIN inputevents_mv v ON c.hadm_id = v.hadm_id 
    AND v.itemid IN (221906) -- Noradrenalin ItemID in Metavision
GROUP BY c.gender;
""")

with engine.connect() as conn:
    df_vaso = pd.read_sql(query_vaso_check, conn)

print(df_vaso)

  gender  gesamt_patienten  patienten_mit_vaso  vaso_rate_prozent
0      F              3471                 566              16.31
1      M              4021                 791              19.67


In [3]:
query_prone_analysis = text("""
WITH first_prone AS (
    SELECT 
        ce.hadm_id,
        MIN(ce.charttime) as first_prone_time
    FROM chartevents ce
    WHERE ce.itemid = 224093 
      AND ce.value = 'Prone'
    GROUP BY ce.hadm_id
)
SELECT 
    c.hadm_id,
    c.gender,
    c.age,
    c.admittime,
    p.first_prone_time,
    -- Zeitdifferenz in Stunden von Aufnahme bis Bauchlage
    EXTRACT(EPOCH FROM (p.first_prone_time - c.admittime))/3600 AS hours_until_prone,
    s.sofa_score,
    s.respiration AS sofa_respiration
FROM cohort_respiratory c
LEFT JOIN first_prone p ON c.hadm_id = p.hadm_id
LEFT JOIN sofa s ON c.hadm_id = s.hadm_id
WHERE p.first_prone_time IS NOT NULL;
""")

import pandas as pd
with engine.connect() as conn:
    df_prone = pd.read_sql(query_prone_analysis, conn)

ProgrammingError: (psycopg2.errors.UndefinedColumn) FEHLER:  Spalte s.respiration existiert nicht
LINE 20:     s.respiration AS sofa_respiration
             ^

[SQL: 
WITH first_prone AS (
    SELECT 
        ce.hadm_id,
        MIN(ce.charttime) as first_prone_time
    FROM chartevents ce
    WHERE ce.itemid = 224093 
      AND ce.value = 'Prone'
    GROUP BY ce.hadm_id
)
SELECT 
    c.hadm_id,
    c.gender,
    c.age,
    c.admittime,
    p.first_prone_time,
    -- Zeitdifferenz in Stunden von Aufnahme bis Bauchlage
    EXTRACT(EPOCH FROM (p.first_prone_time - c.admittime))/3600 AS hours_until_prone,
    s.sofa_score,
    s.respiration AS sofa_respiration
FROM cohort_respiratory c
LEFT JOIN first_prone p ON c.hadm_id = p.hadm_id
LEFT JOIN sofa s ON c.hadm_id = s.hadm_id
WHERE p.first_prone_time IS NOT NULL;
]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [3]:
from sqlalchemy import text  # <--- Diese Zeile hat gefehlt!
import pandas as pd

query_check_scores = text("""
SELECT 
    COUNT(c.hadm_id) as gesamt_anzahl_aki,
    -- SOFA Abdeckung
    SUM(CASE WHEN s.sofa_score IS NOT NULL THEN 1 ELSE 0 END) as mit_sofa,
    ROUND(AVG(s.sofa_score), 2) as avg_sofa_aki,
    -- SAPS II Abdeckung
    SUM(CASE WHEN sap.sapsii_score IS NOT NULL THEN 1 ELSE 0 END) as mit_saps,
    ROUND(AVG(sap.sapsii_score), 2) as avg_saps_aki,
    -- Sterblichkeit nach Score-Verfügbarkeit (Mortalität)
    ROUND(AVG(CASE WHEN s.sofa_score IS NOT NULL THEN c.outcome_death ELSE NULL END) * 100, 2) as mortalitaet_mit_score_prozent
FROM cohort_aki c
LEFT JOIN sofa s ON c.hadm_id = s.hadm_id
LEFT JOIN sapsii sap ON c.hadm_id = sap.hadm_id;
""")

# Ausführung im Notebook
with engine.connect() as conn:
    df_check = pd.read_sql(query_check_scores, conn)

print(df_check)

ProgrammingError: (psycopg2.errors.UndefinedColumn) FEHLER:  Spalte sap.hadm_id existiert nicht
LINE 14: LEFT JOIN sapsii sap ON c.hadm_id = sap.hadm_id;
                                             ^
HINT:  Vielleicht wurde beabsichtigt, auf die Spalte »s.hadm_id« zu verweisen.

[SQL: 
SELECT 
    COUNT(c.hadm_id) as gesamt_anzahl_aki,
    -- SOFA Abdeckung
    SUM(CASE WHEN s.sofa_score IS NOT NULL THEN 1 ELSE 0 END) as mit_sofa,
    ROUND(AVG(s.sofa_score), 2) as avg_sofa_aki,
    -- SAPS II Abdeckung
    SUM(CASE WHEN sap.sapsii_score IS NOT NULL THEN 1 ELSE 0 END) as mit_saps,
    ROUND(AVG(sap.sapsii_score), 2) as avg_saps_aki,
    -- Sterblichkeit nach Score-Verfügbarkeit (Mortalität)
    ROUND(AVG(CASE WHEN s.sofa_score IS NOT NULL THEN c.outcome_death ELSE NULL END) * 100, 2) as mortalitaet_mit_score_prozent
FROM cohort_aki c
LEFT JOIN sofa s ON c.hadm_id = s.hadm_id
LEFT JOIN sapsii sap ON c.hadm_id = sap.hadm_id;
]
(Background on this error at: https://sqlalche.me/e/20/f405)